# **Question No 02**

**AI-Powered Emergency Incident Reporting Assistant**

**a) Structured Prompt — Industrial Safety Incident Analyst**

In [10]:
from langchain_core.prompts import ChatPromptTemplate

detailed_prompt = ChatPromptTemplate.from_template("""
You are an Industrial Safety Incident Analyst working for a large
industrial facility.

Your task is to analyze the employee's incident description and convert
it into a professional and concise safety report.

INCIDENT DESCRIPTION:
{incident_description}

Analyze the incident using only the information provided.

Return the response using EXACTLY these six fields:

1. Incident Category:
   Identify the most appropriate category, such as Slip/Fall,
   Equipment Failure, Fire, Chemical Exposure, Electrical,
   Workplace Injury, Unsafe Condition, or Other.

2. Severity Level:
   Select exactly one:
   Low / Medium / High / Critical

3. Short Incident Summary:
   Provide a concise professional summary of what happened.

4. Possible Immediate Risk:
   Identify the immediate safety risk that may exist.

5. Recommended Immediate Action:
   Recommend practical immediate safety actions.

6. Management Note:
   Provide a brief professional note for management.

CONSTRAINTS:
- Do not invent facts that are not present in the incident description.
- Do not assume injuries, damage, causes, or circumstances that were not stated.
- If information is missing, explicitly state "Not specified in the report."
- Keep the response professional, factual and concise.
- Safety must be prioritized.
- Return all six fields even if some information is unavailable.
""")

**b) LangChain Workflow in Google Colab**

In [3]:
!pip install -q langchain langchain-groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.9 MB/s eta 0:00:00


In [4]:
import os

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

In [5]:
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


In [30]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=1024
)

In [31]:
import groq

client = groq.Groq(api_key=os.environ["GROQ_API_KEY"])
models = client.models.list()

print("Available Groq Models:")
for model in models.data:
    print(f"- {model.id}")

Available Groq Models:
- meta-llama/llama-prompt-guard-2-86m
- groq/compound
- whisper-large-v3-turbo
- openai/gpt-oss-120b
- openai/gpt-oss-20b
- openai/gpt-oss-safeguard-20b
- canopylabs/orpheus-arabic-saudi
- groq/compound-mini
- canopylabs/orpheus-v1-english
- allam-2-7b
- whisper-large-v3
- meta-llama/llama-prompt-guard-2-22m
- qwen/qwen3.6-27b


In [32]:
prompt = ChatPromptTemplate.from_template("""
You are an Industrial Safety Incident Analyst.

Analyze the following employee incident description.

INCIDENT DESCRIPTION:
{incident_description}

Return EXACTLY these fields:

Incident Category:
Severity Level:
Short Incident Summary:
Possible Immediate Risk:
Recommended Immediate Action:
Management Note:

Rules:
- Severity must be Low, Medium, High, or Critical.
- Use only information provided.
- Do not invent facts.
- If information is unavailable, write "Not specified in the report."
- Keep the report professional, concise and safety-focused.
""")

In [33]:
chain = detailed_prompt | llm

Input
  ↓
ChatPromptTemplate
  ↓
LLM
  ↓
Structured Safety Report

In [34]:
incident = """
An employee noticed that water was leaking from a pipe near the
entrance of the production area. No one was injured, but the floor
became wet.
"""

response = chain.invoke({
    "incident_description": incident
})

print(response.content)

**1. Incident Category:** Unsafe Condition  

**2. Severity Level:** Low  

**3. Short Incident Summary:** An employee observed water leaking from a pipe near the entrance to the production area, resulting in a wet floor. No injuries were reported.  

**4. Possible Immediate Risk:** Slip hazard for personnel due to the wet floor.  

**5. Recommended Immediate Action:**  
- Place wet‑floor warning signs around the affected area.  
- Contain and mop up the water promptly.  
- Shut off the water supply to the leaking pipe and notify maintenance for repair.  
- Verify that the area is dry before normal operations resume.  

**6. Management Note:** Prompt repair of the leaking pipe and reinforcement of routine inspections are recommended to prevent recurrence and maintain a safe working environment.


**c) Gradio Interface**

In [35]:
import gradio as gr

def analyze_incident(incident_description):

    if not incident_description.strip():
        return "Please enter an incident description."

    response = chain.invoke({
        "incident_description": incident_description
    })

    return response.content

In [40]:
import gradio as gr

custom_css = """
/* ================================
   GLOBAL APPLICATION
================================ */

body {
    background: #f4f7fb !important;
    color: #1f2937 !important;
    font-family: 'Inter', 'Segoe UI', Arial, sans-serif !important;
}

.gradio-container {
    max-width: 1200px !important;
    margin: 30px auto !important;
    background: #ffffff !important;
    border: 1px solid #e5e7eb !important;
    border-radius: 18px !important;
    box-shadow: 0 10px 35px rgba(15, 23, 42, 0.08) !important;
    overflow: hidden !important;
}


/* ================================
   HEADER
================================ */

h1.gradio-header {
    background: #ffffff !important;
    color: #163b65 !important;
    padding: 30px 25px 12px !important;
    font-size: 2.2rem !important;
    font-weight: 750 !important;
    text-align: center !important;
    border: none !important;
    margin: 0 !important;
    letter-spacing: -0.5px;
}

p.gradio-description {
    background: #ffffff !important;
    color: #64748b !important;
    font-size: 1rem !important;
    line-height: 1.6 !important;
    text-align: center !important;
    padding: 5px 30px 30px !important;
    margin: 0 !important;
}


/* ================================
   INPUT / OUTPUT SECTIONS
================================ */

.gr-block {
    border: none !important;
}

.gr-box {
    background: #ffffff !important;
    border: 1px solid #e2e8f0 !important;
    border-radius: 14px !important;
}


/* ================================
   LABELS
================================ */

.gr-label {
    color: #334155 !important;
    font-size: 0.95rem !important;
    font-weight: 700 !important;
}


/* ================================
   TEXT INPUT
================================ */

.gr-textbox textarea,
.gr-textarea textarea,
.gr-textbox input,
.gr-textarea input {

    background: #f8fafc !important;
    color: #1e293b !important;

    border: 1px solid #cbd5e1 !important;
    border-radius: 12px !important;

    padding: 15px !important;

    font-size: 0.95rem !important;
    line-height: 1.6 !important;

    transition: all 0.25s ease !important;
}


/* Input Focus */

.gr-textbox textarea:focus,
.gr-textarea textarea:focus,
.gr-textbox input:focus,
.gr-textarea input:focus {

    border-color: #2563eb !important;

    box-shadow:
        0 0 0 3px rgba(37, 99, 235, 0.12) !important;

    background: #ffffff !important;
}


/* Placeholder */

.gr-textbox textarea::placeholder,
.gr-textarea textarea::placeholder {

    color: #94a3b8 !important;
}


/* ================================
   PRIMARY BUTTON
================================ */

.gr-button {

    background: #2563eb !important;
    color: #ffffff !important;

    border: none !important;
    border-radius: 10px !important;

    padding: 12px 26px !important;

    font-size: 0.95rem !important;
    font-weight: 700 !important;

    box-shadow:
        0 4px 10px rgba(37, 99, 235, 0.20) !important;

    transition:
        transform 0.2s ease,
        box-shadow 0.2s ease,
        background 0.2s ease !important;
}


/* Button Hover */

.gr-button:hover {

    background: #1d4ed8 !important;

    transform: translateY(-1px) !important;

    box-shadow:
        0 7px 16px rgba(37, 99, 235, 0.25) !important;
}


/* Button Active */

.gr-button:active {

    transform: translateY(0) !important;

}


/* ================================
   MARKDOWN OUTPUT
================================ */

.gr-markdown,
.output-markdown {

    background: #ffffff !important;

    color: #334155 !important;

    border: 1px solid #e2e8f0 !important;

    border-radius: 14px !important;

    padding: 24px !important;

    line-height: 1.7 !important;

    font-size: 0.95rem !important;
}


/* Markdown Headings */

.output-markdown h1,
.output-markdown h2,
.output-markdown h3 {

    color: #163b65 !important;

    font-weight: 750 !important;

    margin-top: 20px !important;
}


/* Markdown Paragraph */

.output-markdown p {

    color: #475569 !important;

}


/* Markdown Lists */

.output-markdown li {

    color: #475569 !important;

    margin-bottom: 6px !important;

}


/* ================================
   TABLES
================================ */

.output-markdown table {

    width: 100% !important;

    border-collapse: collapse !important;

    margin: 20px 0 !important;

    background: #ffffff !important;

    border-radius: 10px !important;

    overflow: hidden !important;
}

.output-markdown th {

    background: #eff6ff !important;

    color: #1e3a5f !important;

    font-weight: 700 !important;

    padding: 12px !important;

    border-bottom: 2px solid #dbeafe !important;
}

.output-markdown td {

    padding: 12px !important;

    color: #475569 !important;

    border-bottom: 1px solid #e5e7eb !important;
}


/* ================================
   BLOCKQUOTES / WARNINGS
================================ */

.output-markdown blockquote {

    background: #fff7ed !important;

    border-left: 4px solid #f59e0b !important;

    color: #92400e !important;

    padding: 12px 16px !important;

    border-radius: 6px !important;

}


/* ================================
   CODE / TECHNICAL CONTENT
================================ */

.output-markdown code {

    background: #f1f5f9 !important;

    color: #334155 !important;

    padding: 3px 6px !important;

    border-radius: 5px !important;

}


/* ================================
   RESPONSIVE DESIGN
================================ */

@media (max-width: 768px) {

    .gradio-container {

        margin: 10px !important;

        border-radius: 12px !important;

    }

    h1.gradio-header {

        font-size: 1.65rem !important;

        padding: 22px 15px 10px !important;

    }

    p.gradio-description {

        font-size: 0.9rem !important;

        padding: 5px 20px 20px !important;

    }

    .gr-textbox textarea {

        min-height: 180px !important;

    }

}


/* ================================
   SCROLLBAR
================================ */

::-webkit-scrollbar {

    width: 8px;

}

::-webkit-scrollbar-track {

    background: #f1f5f9;

}

::-webkit-scrollbar-thumb {

    background: #cbd5e1;

    border-radius: 10px;

}

::-webkit-scrollbar-thumb:hover {

    background: #94a3b8;

}
"""


demo = gr.Interface(

    fn=analyze_incident,

    inputs=gr.Textbox(
        lines=8,
        placeholder=(
            "Describe the workplace incident here...\n\n"
            "Example: During the night shift, an employee observed "
            "smoke and sparks coming from an electrical control panel "
            "in the production area."
        ),
        label="Incident Description"
    ),

    outputs=gr.Markdown(
        label="Safety Incident Report"
    ),

    title="AI-Powered Emergency Incident Reporting Assistant",

    description=(
        "Describe the workplace incident in your own words. "
        "The AI will analyze the event and generate a structured, "
        "professional safety incident report."
    ),

    submit_btn="Analyse Incident",

    clear_btn="Clear",

    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="slate",
        neutral_hue="slate"
    ),

    css=custom_css
)

demo.launch()

/usr/local/lib/python3.13/dist-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  super().__init__(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b2ec638e1e76f8b26c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**d) Testing with Two Incidents
Test Case 1 — Minor Workplace Issue**

Input

An employee noticed that one of the lights in the warehouse is not
working properly. The area is still accessible and no one was injured.

Expected Generated Output

Incident Category: Electrical / Unsafe Condition

Severity Level: Low

Short Incident Summary:
A warehouse light was reported as malfunctioning. No injuries were
reported.

Possible Immediate Risk:
Reduced visibility in the affected warehouse area may create a
potential workplace hazard.

Recommended Immediate Action:
Report the faulty light to maintenance and arrange for its repair.
Restrict or monitor the affected area if visibility becomes unsafe.

Management Note:
Maintenance should inspect and repair the light to prevent the
condition from developing into a safety risk.

**Test Case 2 — High-Risk Safety Incident**

Input

During the night shift, an employee reported seeing smoke coming from
an electrical control panel in the production area. Sparks were visible
and employees were working nearby. No injuries have been reported.

Expected Generated Output

Incident Category: Electrical / Potential Fire

Severity Level: High

Short Incident Summary:
Smoke and visible sparks were reported coming from an electrical
control panel in the production area while employees were nearby.

Possible Immediate Risk:
There is a potential risk of electrical fire, electric shock, and
further equipment damage.

Recommended Immediate Action:
Immediately keep employees away from the affected area and notify
qualified safety and electrical personnel. Follow the facility's
emergency electrical and fire-safety procedures.

Management Note:
The incident requires immediate safety assessment and investigation
before the affected equipment or area is returned to normal operation.

             Employee
                │
                ▼
     ┌─────────────────────┐
     │ Incident Description │
     └──────────┬──────────┘
                │
                ▼
     ┌─────────────────────┐
     │ LangChain Prompt     │
     │ ChatPromptTemplate   │
     └──────────┬──────────┘
                │
                ▼
     ┌─────────────────────┐
     │       LLM           │
     │   Groq / Llama      │
     └──────────┬──────────┘
                │
                ▼
     ┌─────────────────────┐
     │ Structured Safety   │
     │ Report              │
     └──────────┬──────────┘
                │
                ▼
     ┌─────────────────────┐
     │ Gradio Interface    │
     └─────────────────────┘